# Supplementary Code
## Regression performance metrics with 95% bootstrap confidence intervals

This notebook evaluates the predicted bone mineral density (BMD) against the measured BMD in the dataset(.csv file). It reports MAE, MSE, RMSE, R2 and Pearson's correlation coefficient, each with a bootstrap 95% confidence interval.

### 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 2. Load data

In [ ]:
df = pd.read_csv('.csv')
len(df)

### 3. Prepare ground-truth and predicted values

In [ ]:
y_true = np.array(list(df['Total_BMD']))
y_pred = np.array(list(df['predict_bmd']))

### 4. Metric functions

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_func, n_iterations=1000, ci=0.95):
    """Estimate a bootstrap confidence interval for a single metric.

    Parameters
    ----------
    y_true : array-like
        Ground-truth values.
    y_pred : array-like
        Predicted values.
    metric_func : callable
        Function ``metric_func(y_true, y_pred)`` returning a scalar score.
    n_iterations : int, default=1000
        Number of bootstrap resamples.
    ci : float, default=0.95
        Confidence level.

    Returns
    -------
    tuple of float
        (lower_bound, upper_bound) of the confidence interval.
    """
    scores = []
    size = len(y_true)

    for _ in range(n_iterations):
        # Resample indices with replacement
        indices = np.random.randint(0, size, size)
        score = metric_func(y_true[indices], y_pred[indices])
        scores.append(score)

    lower = np.percentile(scores, ((1 - ci) / 2) * 100)
    upper = np.percentile(scores, (1 - (1 - ci) / 2) * 100)

    return lower, upper


def performance(y_true, y_pred, n_bootstrap=1000):
    """Compute regression metrics with their 95% bootstrap confidence intervals.

    Parameters
    ----------
    y_true : array-like
        Ground-truth values.
    y_pred : array-like
        Predicted values.
    n_bootstrap : int, default=1000
        Number of bootstrap resamples.

    Returns
    -------
    dict
        Metric name -> {'value': point estimate, '95%_CI': (low, high)}.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Point estimates
    correlation, _ = pearsonr(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    # Metric functions used for the bootstrap loop
    def calc_correlation(y_t, y_p):
        return pearsonr(y_t, y_p)[0]

    def calc_rmse(y_t, y_p):
        return np.sqrt(mean_squared_error(y_t, y_p))

    # Confidence intervals
    mae_ci = bootstrap_ci(y_true, y_pred, mean_absolute_error, n_bootstrap)
    mse_ci = bootstrap_ci(y_true, y_pred, mean_squared_error, n_bootstrap)
    rmse_ci = bootstrap_ci(y_true, y_pred, calc_rmse, n_bootstrap)
    r2_ci = bootstrap_ci(y_true, y_pred, r2_score, n_bootstrap)
    cc_ci = bootstrap_ci(y_true, y_pred, calc_correlation, n_bootstrap)

    return {
        "MAE": {"value": mae, "95%_CI": mae_ci},
        "MSE": {"value": mse, "95%_CI": mse_ci},
        "RMSE": {"value": rmse, "95%_CI": rmse_ci},
        "R2": {"value": r2, "95%_CI": r2_ci},
        "C.C": {"value": correlation, "95%_CI": cc_ci},
    }

### 5. Performance metrics with 95% confidence intervals

In [ ]:
results = performance(y_true, y_pred)
print(len(y_true))
print(len(y_pred))

for metric, data in results.items():
    print(f"{metric}:")
    print(f"  Value: {data['value']:.4f}")
    print(f"  95% CI: ({data['95%_CI'][0]:.4f}, {data['95%_CI'][1]:.4f})")